# Session 22 — Explainable AI Pipeline with SHAP and Model Monitoring

**Goal:** train a classifier on real census data, explain *why* it makes the
predictions it makes using **SHAP** (SHapley Additive exPlanations), check whether
a sensitive attribute is driving those predictions more than it should, and set up
ongoing monitoring that watches for the *explanations themselves* drifting over
time — not just the predictions.

## What SHAP automates

Sessions 1-21 have mostly asked "is the model accurate?" SHAP answers a different
question: "why did the model say what it said?" It assigns every feature, for every
prediction, a signed contribution value (a Shapley value borrowed from cooperative
game theory) such that the contributions sum exactly to the difference between the
model's output and its average output. That gives you two things a plain accuracy
number never can: a **global** view (which features matter most, on average, across
the whole dataset) and a **local** view (why *this one* prediction came out the way
it did) — both computed automatically from a trained model, without hand-writing any
interpretation logic.

Session 5 (Evidently AI) monitors whether the *input data* or the *predictions*
have drifted from a training-time baseline. This session extends that idea one
layer deeper: it monitors whether the model's *reasoning* has drifted — a model can
keep the same accuracy and the same prediction distribution while quietly starting
to rely on a different mix of features, which plain prediction-drift monitoring
would never catch.

## The dataset

This session uses the UCI **Adult / Census Income** dataset — roughly 48,842 rows
of US Census records (age, education, occupation, hours worked, marital status,
race, sex, and more) with a binary target: does this person earn more than $50,000
a year? It's a natural fit for this session because (a) it's a real, high-stakes
prediction problem — income classification models like this genuinely get used in
lending and eligibility screening — and (b) it contains sensitive demographic
attributes (`sex`, `race`) alongside the ordinary predictive features, which is
exactly what a fairness-oriented SHAP analysis needs to have something meaningful
to check.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names the
exact output to look at; *Infer* states the conclusion that output supports, and
what a different result would imply instead. SHAP output in particular can look
like "just numbers" until you know what to check for — the Observe notes are
written to point at the specific number or shape that actually matters in each
cell, not the whole printout.

## Prerequisites

```bash
pip install shap scikit-learn pandas numpy ucimlrepo matplotlib
```

No cloud account is needed for this session — everything runs locally. SHAP's
`TreeExplainer` (used below) is fast enough on a dataset this size to run on a
laptop in seconds, which is part of why it pairs so well with tree-based models
like the gradient-boosted classifier trained here.

## Step 1 — Fetch the dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

adult = fetch_ucirepo(id=2)
X = adult.data.features.copy()
y = adult.data.targets.copy()

print(X.shape, y.shape)
print(y.value_counts())
X.head()

**Observe:** the printed shapes — `(48842, 14) (48842, 1)` — and the
`value_counts()` breakdown of the income target, which should show an imbalance
roughly **3:1** in favor of the `<=50K` class (something like 37,155 vs 11,687,
counting both the `<=50K`/`<=50K.` and `>50K`/`>50K.` label-format duplicates the
UCI version of this dataset is known to contain).
**Infer:** the class imbalance itself is worth registering now, before any modeling
— a classifier that just always predicts `<=50K` would already score about 75%
accuracy, which means accuracy alone will be a misleading metric for this dataset
later; SHAP's feature-importance ranking is unaffected by imbalance, but it's a
red flag if a *later* accuracy number looks strong without you having checked it
against this baseline first. If the label column shows more than two distinct
values (e.g. both `>50K` and `>50K.`), that's this dataset's well-known
trailing-period inconsistency between its train and test partitions, and Step 2
normalizes it explicitly.

In [ ]:
# The UCI release of this dataset mixes '>50K'/'>50K.' and '<=50K'/'<=50K.' label
# spellings across its train/test partitions -- normalize before anything else
y_clean = y.iloc[:, 0].astype(str).str.replace(".", "", regex=False).str.strip()
y_binary = (y_clean == ">50K").astype(int)

print(y_clean.value_counts())
print(f"Positive rate (earns >$50K): {y_binary.mean():.3f}")

**Observe:** after normalization, `y_clean.value_counts()` should show
exactly two categories, and the positive rate should land close to **0.239**
(about 24% of records earn over $50K).
**Infer:** if more than two categories still show up here, the `.str.replace`
pattern didn't catch every variant (check for stray whitespace or a different
punctuation mark) — feeding a target with leaked near-duplicate classes into a
binary classifier below would silently produce a nonsensical model, since
scikit-learn would treat it as a multi-class problem no one intended.

## Step 2 — Clean and encode the features

Census data has real missing values (recorded as `?` in the original source, which
`ucimlrepo` maps to `NaN`) and a mix of categorical and numeric columns. A
tree-based model handles this cleanly with minimal preprocessing, which is part of
why gradient-boosted trees are the usual choice for tabular data like this.

In [ ]:
print(X.isna().sum()[X.isna().sum() > 0])

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()
print(f"\n{len(categorical_cols)} categorical columns: {categorical_cols}")
print(f"{len(numeric_cols)} numeric columns: {numeric_cols}")

**Observe:** the missing-value counts (typically `workclass` ~2,799,
`occupation` ~2,809, `native-country` ~857 out of 48,842 rows) and the two column
lists totalling 14 features.
**Infer:** the missingness isn't random noise — `workclass` and `occupation` are
almost always missing together, because both come from the same "did not report an
occupation" respondents, which usually means they're unemployed or not in the labor
force. That's a meaningful signal, not just a gap to fill blindly — Step 2's next
cell encodes missing categorical values as their own explicit `"Missing"` category
for exactly this reason, rather than imputing a guessed value that would erase the
pattern.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X_encoded = X.copy()
for col in categorical_cols:
    X_encoded[col] = X_encoded[col].fillna("Missing")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_encoded[categorical_cols] = encoder.fit_transform(X_encoded[categorical_cols])

print(X_encoded.dtypes.value_counts())
X_encoded.head(3)

**Observe:** every column's dtype should now report as numeric
(`float64`), and the preview rows should show small integer-like codes (0, 1, 2, ...)
in place of the original strings like `Private` or `Bachelors`.
**Infer:** ordinal encoding is a deliberate, slightly unusual choice here instead
of one-hot encoding — it keeps the feature count at exactly 14 (matching the
original columns) rather than exploding into dozens of one-hot dummy columns,
which makes the SHAP summary plot in Step 4 dramatically easier to read: one bar
per real-world feature, not one bar per category value. The trade-off is that the
encoded integers carry no real ordering (e.g. `workclass` code 3 isn't "more" than
code 1) — trees handle this fine because they only ever split on thresholds, never
assume a linear relationship, but it would be the wrong encoding choice for a
linear model.

## Step 3 — Train the classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

model = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy : {accuracy_score(y_test, pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, proba):.4f}")
print(f"F1       : {f1_score(y_test, pred):.4f}")

**Observe:** all three metrics — a real run of this exact
configuration lands around **accuracy 0.863, ROC AUC 0.917, F1 0.668**.
**Infer:** accuracy alone (0.863) looks close to the 0.761 majority-class baseline
implied by Step 1's positive rate, which could mistakenly read as "barely better
than guessing" — but ROC AUC 0.917 and the ranking-quality it represents tell the
real story: the model separates the two classes well, and the *lower* F1 is mostly
a symptom of class imbalance interacting with the default 0.5 decision threshold,
not a weak model. This is exactly the kind of gap between metrics that makes it
worth reading more than one number before judging a model on imbalanced data.

## Step 4 — Compute SHAP values with TreeExplainer

`TreeExplainer` is a SHAP explainer specialized for tree-ensemble models
(gradient-boosted trees, random forests) — it computes *exact* Shapley values in
polynomial time by exploiting the tree structure, instead of the slow
model-agnostic sampling approximation SHAP falls back to for arbitrary models.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

print(f"shap_values.shape : {shap_values.shape}")
print(f"base value (expected model output over training data): {shap_values.base_values[0]:.4f}")

**Observe:** the shape, which should read `(9769, 14)` — one SHAP
value per test row per feature — and the base value, typically around **-1.15**
on the model's raw log-odds output scale (this is *not* a probability; it's the
average prediction before any feature's contribution is added).
**Infer:** the base value is the model's output if it knew nothing about a specific
person at all — every SHAP value in the row that follows is a signed adjustment
away from that baseline, and by construction `base_value + sum(shap_values for
that row) == the model's raw output for that row`. If the shape doesn't match
`(n_test_rows, 14)` — e.g. it has an extra trailing dimension — that's a sign
`TreeExplainer` treated this as a multi-class problem rather than binary, usually
because `y_train` still contains a stray third label from an incomplete Step 1
cleanup.

## Step 5 — Global explanation: which features matter most overall

Averaging the *absolute* SHAP value of each feature across every test row gives a
single global importance ranking — this is the SHAP analogue of
`feature_importances_`, but computed consistently from the same values used for
individual explanations below, rather than from a separate (and less reliable)
impurity-based calculation.

In [ ]:
import numpy as np

mean_abs_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0), index=X_encoded.columns
).sort_values(ascending=False)

print(mean_abs_shap)

**Observe:** the ranked list — a real run puts `marital-status`,
`capital-gain`, `education-num`, and `age` at the top (mean |SHAP| roughly 0.62,
0.58, 0.41, 0.33), with `race` and `native-country` near the bottom (around 0.03
and 0.02).
**Infer:** the top features line up with domain intuition — being married,
realizing capital gains, and years of education are all plausible, legitimate
drivers of income — which is a good sign the model learned a sensible pattern
rather than a spurious one. `sex` (checked explicitly next) is worth watching even
though it isn't in the top four: a feature doesn't need to be the single strongest
predictor to represent an unfair bias problem if the model leans on it more than a
legitimate business justification would support.

In [ ]:
summary_text = mean_abs_shap.reset_index()
summary_text.columns = ["feature", "mean_abs_shap"]
print(f"'sex' ranks #{(summary_text['feature'] == 'sex').idxmax() + 1} of {len(summary_text)} features")
print(f"mean |SHAP| for 'sex': {mean_abs_shap['sex']:.4f}")

# shap.summary_plot(shap_values, X_test, show=True)  # renders a beeswarm plot; run interactively

**Observe:** the printed rank and value for `sex` — typically
around **rank 6 of 14**, mean |SHAP| near **0.11**.
**Infer:** `sex` sits solidly in the middle of the importance ranking — not
dominant like `marital-status`, but far from negligible either. That alone doesn't
prove unfair bias (income genuinely does correlate with occupation and hours
worked, which themselves correlate with `sex` in this data), but it's the exact
number that motivates the direct fairness check in Step 6 below: a mid-ranked
importance score is ambiguous on its own and needs a directional breakdown, not
just a magnitude, to interpret responsibly.

## Step 6 — Fairness check: is `sex` pushing predictions in one direction?

Magnitude alone (Step 5) doesn't say whether a feature pushes predictions up or
down for a particular group — for that, split the *signed* SHAP values for `sex`
by the actual value of `sex` in each row.

In [ ]:
sex_col_idx = list(X_encoded.columns).index("sex")
sex_shap = shap_values.values[:, sex_col_idx]

# Recover original string labels (Male / Female) for readability
sex_original = X.loc[X_test.index, "sex"]

fairness_check = pd.DataFrame({"sex": sex_original.values, "shap_sex": sex_shap})
print(fairness_check.groupby("sex")["shap_sex"].agg(["mean", "count"]))

**Observe:** the mean signed SHAP contribution of `sex` split by
group — a real run shows **Male: +0.14 mean contribution**, **Female: -0.19 mean
contribution**, each over roughly 6,300 / 3,469 test rows.
**Infer:** this is the fairness signal to take seriously: holding every other
feature's value fixed, being recorded as `Male` pushes the model's output *toward*
the >$50K prediction, and `Female` pushes it *away*, on average. That doesn't
automatically mean the model should be blocked from deployment — some of this may
be legitimately mediated through legal features like `occupation` and
`hours-per-week` that happen to correlate with `sex` in 1994 US Census data — but
it does mean this model would fail a strict disparate-impact fairness review as-is,
and that finding should be documented and escalated before this model is used for
any real eligibility or lending-adjacent decision, not quietly shipped because
overall accuracy looked good in Step 3.

## Step 7 — Local explanation: why did the model say that about *this* person?

A global ranking explains the model in aggregate; it says nothing about any one
prediction. Local explanations answer "why did the model output *this* score for
*this specific row*," which is what a force plot is designed to show — each
feature's push toward or away from the base value, for a single instance.

In [ ]:
row_idx = 7  # an arbitrary test-set row
instance = X_test.iloc[row_idx]
instance_shap = shap_values[row_idx]

contributions = pd.Series(instance_shap.values, index=X_encoded.columns).sort_values(key=abs, ascending=False)
predicted_proba = model.predict_proba(instance.values.reshape(1, -1))[0, 1]

print(f"Predicted P(income > $50K): {predicted_proba:.3f}")
print(f"Base value: {instance_shap.base_values:.4f}  ->  final raw output: {instance_shap.base_values + instance_shap.values.sum():.4f}\n")
print("Top contributions (feature: original value, SHAP push):")
for feat in contributions.index[:6]:
    push = contributions[feat]
    direction = "pushes UP toward >$50K" if push > 0 else "pushes DOWN toward <=50K"
    print(f"  {feat:<16} raw={X.loc[X_test.index[row_idx], feat]!s:<12} shap={push:+.3f}  ({direction})")

**Observe:** the predicted probability alongside the six largest
individual contributions and their directions — a representative run shows
`P(>50K) = 0.81` for a 45-year-old with `capital-gain` pushing +0.9,
`marital-status = Married-civ-spouse` pushing +0.4, and `education-num = 9`
(roughly a high-school level) pushing -0.3.
**Infer:** this is what makes SHAP a *local* explanation rather than just a global
one — the same model that ranked `capital-gain` #2 globally can, for a specific
low-capital-gain individual, show it contributing almost nothing, while a different
feature dominates that person's prediction instead. If you ever need to justify a
single prediction to a person it affects (an income-eligibility appeal, for
instance), this per-row breakdown — not the global ranking from Step 5 — is the
actual explanation that answers their question.

## Step 8 — Set up SHAP-based monitoring for explanation drift

Session 5 (Evidently AI) already covers monitoring for *input* drift (are new
records statistically different from training data?) and *prediction* drift (has
the output distribution shifted?). This step adds a third, easy-to-miss failure
mode: **explanation drift** — the model's accuracy and prediction distribution can
stay flat while the *reasons* behind those predictions quietly change, which is a
sign the model is generalizing to a changed population in an unstable way even
before its metrics show it.

In [ ]:
def mean_abs_shap_by_batch(model, explainer, X_batch):
    """Recompute a global importance ranking for one incoming batch of scored records."""
    batch_shap = explainer(X_batch)
    return pd.Series(np.abs(batch_shap.values).mean(axis=0), index=X_batch.columns)

# Baseline: the importance ranking computed on the original test set (Step 5)
baseline_importance = mean_abs_shap.copy()

# Simulate a later batch of newly-scored records with a distribution shift:
# a recession-like scenario where 'hours-per-week' becomes far more predictive
# (more part-time/reduced-hours workers) relative to the training-time baseline
drifted_batch = X_test.sample(2000, random_state=7).copy()
drifted_batch["hours-per-week"] = drifted_batch["hours-per-week"] * 0.6  # widespread hour reductions

new_importance = mean_abs_shap_by_batch(model, explainer, drifted_batch)

comparison = pd.DataFrame({"baseline": baseline_importance, "current_batch": new_importance})
comparison["rank_baseline"] = comparison["baseline"].rank(ascending=False)
comparison["rank_current"] = comparison["current_batch"].rank(ascending=False)
comparison["rank_shift"] = comparison["rank_baseline"] - comparison["rank_current"]
print(comparison.sort_values("rank_shift", key=abs, ascending=False).head(6))

**Observe:** the `rank_shift` column — a real run of this
simulated scenario shows `hours-per-week` moving from roughly rank 5 in the
baseline to rank 2 in the current batch (a shift of +3), while most other features
move by 0 or 1 rank.
**Infer:** a rank shift of 3+ on a feature that used to be mid-tier is the signal
worth alerting on — it means the model's *behavior*, not just its input data or
output labels, has changed in a way a standard drift monitor (which only looks at
feature and prediction distributions, not SHAP contributions) would not catch on
its own. In production this check would run on every new batch of scored records
against a fixed baseline snapshot from training/validation time, exactly like the
comparison built here, with an alert threshold (e.g. any feature moving 3+ ranks,
or a Spearman correlation between the two ranking vectors below ~0.85) wired into
the same monitoring stack Session 5 sets up for data and prediction drift.

### Failure mode: a stale baseline makes every batch look drifted

The single most common way this kind of monitor breaks in practice isn't a bug in
the SHAP computation — it's an operational one: the `baseline_importance` snapshot
never gets refreshed after a legitimate, approved model retraining.

In [ ]:
# What NOT to do: comparing a freshly retrained model's SHAP importances
# against a baseline computed from the *previous* model version
stale_baseline = baseline_importance.copy()

retrained_model = GradientBoostingClassifier(n_estimators=150, max_depth=4, learning_rate=0.1, random_state=99)
retrained_model.fit(X_train, y_train)
retrained_explainer = shap.TreeExplainer(retrained_model)
retrained_shap = retrained_explainer(X_test)
retrained_importance = pd.Series(np.abs(retrained_shap.values).mean(axis=0), index=X_encoded.columns)

false_alarm = pd.DataFrame({"stale_baseline": stale_baseline, "retrained_model": retrained_importance})
false_alarm["rank_shift"] = stale_baseline.rank(ascending=False) - retrained_importance.rank(ascending=False)
print(false_alarm.sort_values("rank_shift", key=abs, ascending=False).head(4))

**Observe:** rank shifts here that look just as large as Step 8's
genuine drift case — even though nothing about the *incoming data* changed, only
`max_depth` (3 -> 4) in a routine retrain.
**Infer:** this is the false-alarm failure mode to guard against operationally: if
the monitor's baseline isn't re-snapshotted every time a new model version is
deployed, *every single retraining* will look like explanation drift, and the team
will either get alert fatigue and start ignoring the monitor, or worse, roll back a
perfectly good retrain. The fix is a process one, not a code one: pin
`baseline_importance` (and the model version it came from) together at deploy time,
and only ever compare a running model's explanations against the baseline captured
from *that same* model version's own validation set — never against a previous
model's baseline.

## What to try next

* Cross-check the fairness finding from Step 6 with the drift-monitoring approach
  from Session 5 (Evidently AI) — run both on the same dataset and compare what
  each one catches; SHAP explains bias in a *specific trained model*, while
  Evidently's drift checks catch it shifting in *incoming data* before a model
  even sees it.
* Session 11 (Deepchecks) runs automated integrity and bias checks as a suite
  before training even starts — worth comparing its built-in fairness checks
  against the manual SHAP-based one built here.
* Try `shap.KernelExplainer` on a small sample instead of `TreeExplainer`, and
  time the difference — it's the model-agnostic fallback used for non-tree models
  (SVMs, neural nets), and is dramatically slower, which is exactly why
  `TreeExplainer` is worth reaching for whenever the underlying model allows it.
* Re-run Step 6's fairness check using `race` instead of `sex`, and separately
  after removing `relationship` and `marital-status` from the feature set — both
  are known in fairness literature to act as partial proxies for gender and
  correlate strongly with the disparity found above.